In [16]:
import json
from pathlib import Path
from datetime import datetime as dt
import pandas as pd
import numpy as np
import altair as alt


def load_theme(path: str | Path) -> dict:
    with Path(path).open() as f:
        return json.load(f)


theme_data = load_theme("data/styles/poliscope_theme.json")


@alt.theme.register("poliscope_theme", enable=True)
def poliscopetheme() -> dict:
    return {**theme_data.get("config", {}), "colors": theme_data.get("colors", {})}

In [17]:
df = pd.read_csv("./data/raw/2026-08-07_hitzeschutz_grouped_matches.csv")
topic = "hitzeschutz"
datestring = dt.now().strftime("%Y-%m-%d")

# Städte mit den meisten Treffern

Erkläuterung zur Differenzierung "Proposal" und "Meeting":
- Ein Meeting ist eine Sitzung, in der das gesuchte Thema behandelt worden ist
- In der Kommunalpolitik sind Beratungen in "Vorgängen" gebündelt. In einigen, aber nicht in allen RIS sind Meetings auch technisch Vorgängen zugeordnet

In [18]:
#Anzahl an Kommunen, für die es Treffer gibt
len(df["entityId"].unique())

269

Das Histogramm zeigt, ob ein Thema überall gleich viel diskutiert wird, oder ob es einzelne Spitzenreiter mit sehr vielen Sitzungen zum Them gibt, aber viele Kommunen mit sehr wenigen Treffern.

In [19]:
entity_counts = (
    df.assign(city=df["entityName"].fillna("Unknown"))
    .groupby("city")
    .size()
    .reset_index(name="match_count")
)

bin_size = 5

bins = np.arange(0, entity_counts["match_count"].max() + bin_size, bin_size)
hist_counts, bin_edges = np.histogram(entity_counts["match_count"], bins=bins)
histogram_df = pd.DataFrame({
    "bin_start": bin_edges[:-1],
    "bin_end": bin_edges[1:],
    "count": hist_counts
})
histogram_df["bin_label"] = histogram_df["bin_start"].astype(int).astype(str) + "–" + histogram_df["bin_end"].astype(int).astype(str)
histogram_df.to_csv(f"./data/processed/{datestring}_histogram_entity_counts_{topic}.csv", index=False)

mean_matches_per_entity = entity_counts["match_count"].mean()
text_color = theme_data.get("config", {}).get("axis", {}).get("titleColor") or theme_data.get("colors", {}).get("brand", {}).get("500")

histogram = (
    alt.Chart(entity_counts)
    .mark_bar(color="#00373a")
    .encode(
        x=alt.X(
            "match_count:Q",
            bin=alt.Bin(step=5),
            title="Treffer pro Kommune"
        ),
        y=alt.Y("count():Q", title="Anzahl Kommunen"),
        tooltip=["match_count:Q", "count():Q"],
    )
    .properties(width=800, height=300, title=f"Verteilung der Treffer pro Kommune (Mittelwert: {mean_matches_per_entity:.2f})")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
)

histogram

alt.Chart(...)

In [20]:
# altair bar chart with the top 20 cities that have the highest count of matches
# the bars are stacked based on the amount of autofrei and autoarm (value in the column class)

#aus den keys können wir ablesen, ob der Datenpunkt ein Proposal oder ein Meeting ist
df["groupType"] = df["groupKey"].astype(str).str.split(":", n=1).str[0]
stack_col = "groupType"
#df für Viz aufbereiten
plot_df = (
    df.assign(city=df["entityName"].fillna("Unknown"))
    .groupby(["city", stack_col], dropna=False)
    .size()
    .reset_index(name="matches")
)

#top 20 Städte identifizieren
city_totals = plot_df.groupby("city")["matches"].sum().sort_values(ascending=False)
city_totals.to_csv(f"./data/processed/{datestring}_city_totals_{topic}.csv", index=True)

top_cities = city_totals.head(20).index.tolist()
plot_df = plot_df[plot_df["city"].isin(top_cities)].copy()

text_color = theme_data.get("config", {}).get("axis", {}).get("titleColor") or theme_data.get("colors", {}).get("brand", {}).get("500")
category_colors = ["#306969", "#4FB0B0"]

chart = (
    alt.Chart(plot_df)
    .mark_bar(size=15)
    .encode(
        y=alt.Y("city:N", title=None, sort=top_cities),
        x=alt.X(
            "matches:Q",
            title="Anzahl Treffer",
            scale=alt.Scale(zero=True)
        ),
        color=alt.Color(
            f"{stack_col}:N",
            title="Kategorie",
            scale=alt.Scale(range=category_colors if category_colors else None)
        ),
        tooltip=["city:N", f"{stack_col}:N", "matches:Q"],
    )
    .properties(width=800, height=400, title="Top 20 Städte nach Trefferanzahl")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
    .configure_legend(labelColor=text_color, titleColor=text_color)
)

chart

alt.Chart(...)

# Treffer über Zeit

Wir betrachten nur die Zeit ab Mitte 2023, weil Poliscope nur bis Afang 2024 einen nahezu vollständigen Datensatz bieten kann. Dieser enthält aber auch in Einzelfällen Verweise auf länger zurückliegende Sitzungen.

In [21]:
subset = df
#optional nur für eine Stadt darstellen
#subset = df[df["entityName"] == "Bochum, Stadt"]

subset["date"] = pd.to_datetime(subset["date"], errors="coerce")
weekly = (
    subset.dropna(subset=["date"])
    .loc[subset["date"] >= "2023-07-01"]
    .assign(week=subset["date"].dt.to_period("W-MON").dt.to_timestamp())
    .groupby("week")
    .size()
    .reset_index(name="matches")
)
weekly = weekly.sort_values("week").reset_index(drop=True)
weekly["week_label"] = weekly["week"].dt.strftime("%Y-%m-%d")
weekly["show_label"] = weekly.index % 3 == 0

weekly.to_csv(f"./data/processed/{datestring}_weekly_matches_{topic}.csv", index=False)

weekly_chart = (
    alt.Chart(weekly)
    .mark_bar(size=2)
    .encode(
        x=alt.X(
            "week:T",
            title="Woche",
            axis=alt.Axis(format="%Y-%m-%d", labelExpr="datum.value % 3 == 0 ? timeFormat(datum.value, '%Y-%m-%d') : ''"),
        ),
        y=alt.Y("matches:Q", title="Anzahl Treffer", scale=alt.Scale(zero=True)),
        color=alt.ColorValue("#00373a"),
        tooltip=["week_label:N", "matches:Q"],
    )
    .properties(width=800, height=300, title="Treffer pro Woche")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
)

weekly_chart

alt.Chart(...)

In [22]:
weekly.to_csv("./data/processed/weekly_matches.csv", index=False)

# Treffer pro Stadt redaktionell prüfen

In [23]:
city = "Bochum, Stadt"

In [24]:
city_matches = df[df["entityName"] == city].copy()
city_matches

,Unnamed: 0,groupKey,date,chunkCount,proposalId,entityId,entityLevel,entityName,groupType


In [25]:
# stacked bars that show the amount of proposals and meetings in each city